## **Notebook 4 - Deep Learning Model using LSTM**

In this notebook, we implement a Deep Learning model using LSTM (Long Short-Term Memory) for disaster tweet classification.

### Objectives:
- Convert text into sequences using Tokenization
- Build an LSTM-based neural network
- Capture sequential dependencies in text
- Compare performance with traditional ML models

This approach helps in understanding contextual relationships in text beyond TF-IDF features.

#### **Import Required Libraries**

In [3]:
import pandas as pd
import numpy as np
import logging


import tensorflow as tf
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

#### **Load Dataset**

In [4]:
df = pd.read_csv("../data/complete_dataset_prepared.csv")
df.head()

,id,keyword,location,text,clean_text,tokens,text_length_chars,word_count,has_hashtag,has_mention,has_url,target
0,1,missing,missing,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...,"['our', 'deeds', 'are', 'the', 'reason', 'of',...",69,13,True,False,False,1
1,4,missing,missing,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,"['forest', 'fire', 'near', 'la', 'ronge', 'sas...",38,7,False,False,False,1
2,5,missing,missing,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...,"['all', 'residents', 'asked', 'to', 'shelter',...",133,22,False,False,False,1
3,6,missing,missing,"13,000 people receive #wildfires evacuation or...",people receive wildfires evacuation orders in ...,"['people', 'receive', 'wildfires', 'evacuation...",65,8,True,False,False,1
4,7,missing,missing,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...,"['just', 'got', 'sent', 'this', 'photo', 'from...",88,16,True,False,False,1


#### **Logger Configuration**

This project uses Python logging instead of print statements.

Why?
- Debugging large datasets
- Tracking preprocessing steps
- Production ML pipelines
- Helps identify where a failure occurred

All activities will be recorded inside:
`model_insights_feature_analysis.log`

In [5]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
LOG_DIR = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "deep_learning_lstm.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8")
    ],
    force=True
)

logger = logging.getLogger(__name__)

logger.info("==== DEEP LEARNING LSTM PIPELINE STARTED ====")
print(f"Logging to: {LOG_FILE}")

Logging to: E:\DData\Projects\DSC\NextHikes\Python\disaster-tweet-classification-nlp-pro-7\logs\deep_learning_lstm.log


---

#### **Define Features and Target**

In [6]:
X = df["clean_text"]
y = df["target"]